# Modelo RNN (LSTM) — falha de porta em 7 dias

O Modelo 5 de Random Forest testavaao  para ver se o histórico de alarmes de uma porta prevê uma falha nos próximos 7 dias, usando features feitas em janelas móveis (1/3/7/14/30 dias). Este notebook testa a mesma pergunta com uma rede neural recorrente (RNN), que recebe uma sequência não processada de alarmes e aprende sozinha o que importa olhar no tempo, sem ter alguém para desenhar as janelas na mão.

## 1. Comparação com o Modelo 5

Foi pensaod que para uma comparação entre dois modelos fazer sentido, os dois precisam responder à mesma pergunta com mesmo alvo e mesmas métricas. Assim a diferença pode ser atribuida às diferentes arquiteturas.

Este modelo usa a sequência de alarmes por porta específica, a mesma unidade de análise do Modelo 5 (porta-dia) e o mesmo alvo (`y = 1` se houve uma ordem corretiva emergencial, ZM01, nos 7 dias seguintes). Assim a comparação é feita com o Modelo 5, os dois preveem  a mesma coisa, só que com estruturas diferentes.

## 2. O que é uma Rede Neural Recorrente (RNN)

O Random Forest recebe uma linha por porta por dia com números já calculados; quantos alarmes teve nos últimos 7 dias, nos últimos 30, etc. Cada uma dessas colunas foi pensada antes de treinar o modelo.

Porém com uma RNN esse processo é diferente: ela recebe a sequência de dias, um de cada vez, na ordem em que aconteceram, e processa cada dia levando em conta o que ela "lembra" dos dias anteriores. Ao processar o dia 2, ela já sabe o que viu no dia 1; ao processar o dia 3, ela já sabe o que aconteceu nos dias 1 e 2, e assim por diante até o último dia da sequência (nesse caso, os 30 dias antes de `t0`). Esse resumo do que já viu até agora se chama hidden state, e é ele que carrega a memória de um dia pro outro.

No fim da sequência, o hidden state guarda um resumo dos 30 dias, aprendido pela própria rede durante o treino, e não desenhado à mão como as janelas móveis do Modelo 5. Uma última camada transforma esse resumo em uma probabilidade de falha.

## 3. Por que LSTM em vez de uma RNN simples

O problema de uma RNN simples é que, conforme a sequência cresce, a rede vai esquecendo o que viu muitos passos atrás, ou seja informação do dia 1 praticamente some do hidden state até chegar no dia 30.

**LSTM** (Long Short-Term Memory) resolve isso adicionando uma segunda memória, mais estável, que passa de um dia pro outro quase sem mudar, a não ser que a rede decida explicitamente atualizar ou apagar parte dela. Ela faz isso com três gates, que a própria rede aprende a usar durante o treino:

- **portão de esquecimento:** decide o que apagar da memória;
- **portão de entrada:** decide o que adicionar do dia atual;
- **portão de saída:** decide o que passar adiante pro próximo dia.

Essas decisões não são programadas, a rede aprende os pesos desses portões olhando para os dados de treino, do mesmo jeito que o Random Forest aprende onde cortar cada árvore olhando pros dados. Na prática, isso significa que a LSTM consegue notar, por exemplo, que uma porta teve um alarme importante há 3 semanas mesmo que os dias mais recentes estejam calmos, algo que uma RNN simples teria "esquecido".

## 4. Como as sequências foram montadas

As sequências foram montadas de forma que:

- No Modelo 5, cada observação era uma linha, representando uma porta em um dia específico.
- Na RNN, cada observação vira uma sequência dos últimos 30 dias daquela porta, em ordem cronológica.

Ou seja para uma porta específica, por exemplo `BUT_1_08`, e um dia de previsão `t0 = 15/06/2024`.

No Modelo 5, a linha daquela porta naquele dia seria algo como:

| porta | data | n_CA_7d | n_CA_30d | ... | y |
|---|---|---:|---:|---:|---:|
| BUT_1_08 | 15/06/2024 | 3 | 12 | ... | 1 |

Assim, ele recebe um resumo do histórico: “quantos alarmes aconteceram nos últimos 7 dias, 30 dias, etc.”

Já para a RNN, em vez de um resumo, a ideia é usar os dados do passado dia a dia. Então, para a mesma porta e mesmo dia de previsão, montamos uma tabela como esta:

| dia | n_CA | n_FC | ... | n_total |
|---|---:|---:|---:|---:|
| 17/05 | 0 | 1 | ... | 1 |
| 18/05 | 0 | 0 | ... | 0 |
| 19/05 | 2 | 0 | ... | 2 |
| ... | ... | ... | ... | ... |
| 14/06 | 2 | 0 | ... | 2 |

Essa tabela tem:

- 30 linhas, uma para cada dia anterior ao t0
- 12 colunas, uma para cada feature daquele dia

Ou seja, uma amostra da RNN é:

- sequência de 30 dias
- cada dia com 12 features
- organizada na ordem em que os eventos aconteceram

A data de previsão t0 não entra na sequência. A sequência termina no dia anterior, `14/06`, para evitar vazamento de informação do futuro.

Assim, a entrada da RNN tem formato:

- [n_amostras, 30, 12]

Onde:

- 30 = quantidade de dias na sequência
- 12 = quantidade de features por dia

Por isso, o treino da LSTM não recebe uma linha com janelas móveis. Ele recebe o histórico dos últimos 30 dias em sequência, e é a própria rede que aprende quais padrões do tempo importam para prever a falha.

Em resumo:

- Random Forest: uma linha por porta por dia, com features já calculadas
- RNN: uma sequência de 30 dias por porta, com as features diárias em ordem

Essa diferença é justamente o que faz a RNN aprender padrões temporais, em vez de depender só de agregados pré-definidos.


In [1]:
import os
# a instalacao do torch neste ambiente usa uma OpenMP diferente da do numpy/scikit-learn
# (MKL); sem isso, e sem limitar a 1 thread, o treino trava (nao da erro, so nunca avanca)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import numpy as np
import pandas as pd
import torch
torch.set_num_threads(1)
import torch.nn as nn
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              average_precision_score, roc_auc_score, confusion_matrix)

torch.manual_seed(42)
np.random.seed(42)

d = np.load('sequencias.npz')
X, y = d['X'], d['y']
idx = pd.read_parquet('sequencias_index.parquet')
print(f'X: {X.shape} | y: {y.shape} | taxa de positivos: {y.mean()*100:.2f}%')

X: (306864, 30, 12) | y: (306864,) | taxa de positivos: 8.09%


## 5. Separação treino/teste

Foi feito o mesmo corte temporal do Modelo 5. Com treino até 23/04/2025, teste a partir de 01/05/2025 (7 dias de folga entre os dois), e uma parte de validação dentro do treino (a partir de 31/01/2025), que foi usada aqui pra decidir quantas épocas treinar e qual limiar de decisão usar.

In [2]:
FIM_TREINO, INI_TESTE, INI_VALID = (pd.Timestamp('2025-04-23'), pd.Timestamp('2025-05-01'),
                                     pd.Timestamp('2025-01-31'))
data = idx['Data']
m_treino = data <= FIM_TREINO
m_teste = data >= INI_TESTE
m_fit = m_treino & (data <= INI_VALID - pd.Timedelta(days=8))
m_val = m_treino & (data >= INI_VALID)

print(f'fit: {m_fit.sum():,} | val: {m_val.sum():,} | treino: {m_treino.sum():,} | teste: {m_teste.sum():,}')
print(f'taxa falha fit: {y[m_fit.values].mean()*100:.2f}% | val: {y[m_val.values].mean()*100:.2f}% | teste: {y[m_teste.values].mean()*100:.2f}%')

fit: 194,304 | val: 35,520 | treino: 232,992 | teste: 71,184
taxa falha fit: 7.64% | val: 9.28% | teste: 8.71%
